# Vesuvius V9 - Optimized Training (Break 0.550 LB)

## Key Fixes Applied

| Fix | What | Why |
|-----|------|-----|
| **#1** | AdamW + CosineAnnealing | SGD locks into sharp minima; AdamW handles topology gradients better |
| **#2** | PATCHES_PER_VOLUME=12 | More tight folds/damaged regions → better SurfaceDice |
| **#3** | N_BLOCKS=[1,3,5,7,7,7] | Deeper in low-res layers → better global sheet reasoning |
| **#4** | Dice-first, topology-later | First epochs learn surfaces, then topology cleans up |
| **#5** | Overlap=0.5 in inference | Reduces tile artifacts → better VOI |
| **#6** | Post-processing | Largest CC + closing + island removal → +0.01-0.03 LB |

## Loss Schedule (Scaled for 300 epochs per fold)
- **Epochs 0-50**: Dice=0.35, BCE=0.10, Skel=0, clDice=0 (learn clean surfaces)
- **Epochs 50-100**: Dice=0.35, BCE=0.10, Skel=0.30 warmup (add skeleton)
- **Epochs 100-300**: Full topology (Dice=0.35, Skel=0.30, clDice=0.25)

---

## Cross-Validation Strategy (CRITICAL)

### ❌ BAD: Random slice-level CV
- Same physical scroll regions appear in both train and val
- Val Dice will be optimistically biased
- LB will NOT correlate with val

### ✅ GOOD: Volume-level K-Fold CV
- Each fold tests on unseen physical regions
- Much better correlation with LB
- Prevents leakage across adjacent CT slices

### ✅ BEST: Leave-One-Scroll-Out (LOSO)
- Train on all but one scroll
- Validate on held-out scroll
- Gold standard for this competition

## Ensemble Strategy
```
5-Fold CV × 300 epochs = 1500 total epochs (vs 6000 with 1200/fold)
5 checkpoints → Average predictions → Best LB
```

In [ ]:
!pip install imagecodecs connected-components-3d -q

In [ ]:
# =============================================================================
# CELL 1: IMPORTS + CONFIG (STRICT 3-FOLD CV, LEADERBOARD-ALIGNED)
# =============================================================================

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import csv
import sys
import json
import time
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from PIL import Image, ImageSequence
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from skimage.morphology import skeletonize
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')


def build_epoch_schedule(total_epochs: int) -> Dict[str, int]:
    """
    Relative topology schedule.
    Reference schedule at 300 epochs:
      skel_start=45, skel_warmup=45, cldice_start=120, cldice_warmup=60
    """
    if total_epochs <= 0:
        raise ValueError('total_epochs must be > 0')
    scale = total_epochs / 300.0
    skel_start = int(round(45 * scale))
    skel_warmup = max(1, int(round(45 * scale)))
    cldice_start = int(round(120 * scale))
    cldice_warmup = max(1, int(round(60 * scale)))
    # Keep phases ordered
    cldice_start = max(cldice_start, skel_start + 1)
    return {
        'skel_start': skel_start,
        'skel_warmup': skel_warmup,
        'cldice_start': cldice_start,
        'cldice_warmup': cldice_warmup,
    }


@dataclass
class Config:
    # Paths
    DATA_ROOT: Path = Path('/kaggle/input/3d-volume-training-data')
    CHECKPOINT_DIR: Path = Path('/kaggle/working/checkpoints_v9_cv')
    OOF_DIR: Path = Path('/kaggle/working/oof_probs')
    METRICS_DIR: Path = Path('/kaggle/working/metrics_v9_cv')
    FOLD_MANIFEST: Path = Path('/kaggle/working/folds_seed42_n3.json')

    # Model
    PATCH_SIZE: Tuple[int, int, int] = (128, 128, 128)
    FEATURES: List[int] = None
    N_BLOCKS: List[int] = None
    USE_SCSE: bool = True
    USE_DEEP_SUPERVISION: bool = True

    # CV schedule
    N_SPLITS: int = 3
    EPOCHS_PER_FOLD: int = 300
    EARLY_STOP_PATIENCE: int = 60

    # Validation protocol
    PATCH_DICE_EVERY: int = 5
    LB_EVAL_EVERY: int = 20
    LB_EVAL_MAX_CASES: int = 8
    VALIDATION_OVERLAP: float = 0.5

    # Data loading
    BATCH_SIZE: int = 16
    NUM_WORKERS: int = 12
    PREFETCH_FACTOR: int = 4
    PATCHES_PER_VOLUME: int = 16
    VAL_PATCHES_PER_VOLUME: int = 2
    FG_OVERSAMPLE_RATIO: float = 0.60

    # Optimizer / scheduler
    LEARNING_RATE: float = 1.8e-3
    BETA1: float = 0.9
    BETA2: float = 0.95
    WEIGHT_DECAY: float = 1e-4
    ETA_MIN: float = 1e-6
    GRADIENT_CLIP: float = 8.0

    # Loss
    DICE_WEIGHT: float = 0.45
    BCE_WEIGHT: float = 0.20
    SKELETON_WEIGHT: float = 0.20
    CLDICE_WEIGHT: float = 0.15

    # Postprocess defaults for LB eval / OOF
    PP_THRESHOLD: float = 0.30
    PP_USE_HYSTERESIS: bool = True
    PP_HYST_LOW: float = 0.16
    PP_HYST_HIGH: float = 0.38
    PP_MIN_COMPONENT_SIZE: int = 50
    PP_CLOSING_ITERS: int = 0

    # Metric constants
    SPACING: Tuple[float, float, float] = (1.0, 1.0, 1.0)
    SURFACE_TOLERANCE: float = 2.0
    IGNORE_LABEL: int = 2
    VOI_CONNECTIVITY: int = 26

    # Runtime
    DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    USE_AMP: bool = True
    SEED: int = 42

    # Non-CV training is intentionally disabled
    ENABLE_NON_CV_ENTRYPOINT: bool = False

    def __post_init__(self):
        if self.FEATURES is None:
            self.FEATURES = [32, 64, 128, 256, 320, 320]
        if self.N_BLOCKS is None:
            self.N_BLOCKS = [1, 2, 3, 5, 5, 5]
        self.SCHEDULE = build_epoch_schedule(self.EPOCHS_PER_FOLD)
        self.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        self.OOF_DIR.mkdir(parents=True, exist_ok=True)
        self.METRICS_DIR.mkdir(parents=True, exist_ok=True)


cfg = Config()
cfg.__post_init__()


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True


set_seed(cfg.SEED)

print('=' * 90)
print('V9 STRICT CV CONFIG')
print('=' * 90)
print(f'Device: {cfg.DEVICE} | AMP: {cfg.USE_AMP}')
print(f'Patch={cfg.PATCH_SIZE}, Batch={cfg.BATCH_SIZE}, Patches/vol={cfg.PATCHES_PER_VOLUME}')
print(f'3-fold CV, epochs/fold={cfg.EPOCHS_PER_FOLD}, patience={cfg.EARLY_STOP_PATIENCE}')
print('Schedule:', cfg.SCHEDULE)
print(f'Loss weights: dice={cfg.DICE_WEIGHT}, bce={cfg.BCE_WEIGHT}, skel={cfg.SKELETON_WEIGHT}, cldice={cfg.CLDICE_WEIGHT}')
print(f'Optimizer: AdamW lr={cfg.LEARNING_RATE}, wd={cfg.WEIGHT_DECAY}, betas=({cfg.BETA1}, {cfg.BETA2}), clip={cfg.GRADIENT_CLIP}')
print('Postprocess defaults:', {
    'threshold': cfg.PP_THRESHOLD,
    'hysteresis_low': cfg.PP_HYST_LOW,
    'hysteresis_high': cfg.PP_HYST_HIGH,
    'min_component_size': cfg.PP_MIN_COMPONENT_SIZE,
    'closing_iters': cfg.PP_CLOSING_ITERS,
})
print('=' * 90)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('TF32 enabled')


In [ ]:
# =============================================================================
# CELL 2: MODEL (6-stage ResEncUNet3D with deeper L3-L5)
# =============================================================================

class SafeInstanceNorm3d(nn.Module):
    def __init__(self, num_features, eps=1e-5, affine=True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.weight = nn.Parameter(torch.ones(num_features))
            self.bias = nn.Parameter(torch.zeros(num_features))
    
    def forward(self, x):
        mean = x.mean(dim=[2,3,4], keepdim=True)
        var = x.var(dim=[2,3,4], keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var.clamp(min=self.eps) + self.eps)
        if self.affine:
            x = self.weight.view(1,-1,1,1,1) * x + self.bias.view(1,-1,1,1,1)
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            SafeInstanceNorm3d(out_ch),
            nn.LeakyReLU(0.01, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class ResBlock(nn.Module):
    def __init__(self, channels, n_convs=2):
        super().__init__()
        self.blocks = nn.Sequential(*[ConvBlock(channels, channels) for _ in range(n_convs)])
    
    def forward(self, x):
        return x + self.blocks(x)


class scSEBlock(nn.Module):
    def __init__(self, channels, reduction=2):
        super().__init__()
        self.cse_pool = nn.AdaptiveAvgPool3d(1)
        self.cse_fc1 = nn.Linear(channels, channels // reduction)
        self.cse_fc2 = nn.Linear(channels // reduction, channels)
        self.sse_conv = nn.Conv3d(channels, 1, 1)
    
    def forward(self, x):
        b, c = x.shape[:2]
        cse = torch.sigmoid(self.cse_fc2(F.relu(self.cse_fc1(self.cse_pool(x).view(b,c))))).view(b,c,1,1,1)
        sse = torch.sigmoid(self.sse_conv(x))
        return x * cse + x * sse


class ResEncUNet3D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=None, n_blocks=None,
                 use_scse=True, use_deep_supervision=True):
        super().__init__()
        
        if features is None:
            features = [32, 64, 128, 256, 320, 320]
        if n_blocks is None:
            n_blocks = [1, 3, 5, 7, 7, 7]
        
        self.features = features
        self.use_scse = use_scse
        self.use_deep_supervision = use_deep_supervision
        
        # Encoder
        self.encoders = nn.ModuleList()
        self.attentions = nn.ModuleList()
        self.pools = nn.ModuleList()
        
        for i, (feat, nb) in enumerate(zip(features, n_blocks)):
            in_channels = in_ch if i == 0 else features[i-1]
            self.encoders.append(nn.Sequential(
                ConvBlock(in_channels, feat),
                *[ResBlock(feat) for _ in range(nb)]
            ))
            self.attentions.append(scSEBlock(feat) if use_scse else nn.Identity())
            if i < len(features) - 1:
                self.pools.append(nn.Conv3d(feat, feat, 2, stride=2))
        
        # Decoder
        self.ups = nn.ModuleList()
        self.dec_convs = nn.ModuleList()
        
        for i in range(len(features)-2, -1, -1):
            self.ups.append(nn.ConvTranspose3d(features[i+1], features[i], 2, stride=2))
            self.dec_convs.append(ConvBlock(features[i]*2, features[i]))
        
        self.final = nn.Conv3d(features[0], out_ch, 1)
        
        if use_deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(features[-(i+2)], out_ch, 1) 
                for i in range(min(3, len(features)-1))
            ])
    
    def forward(self, x):
        enc_features = []
        for i, (enc, att) in enumerate(zip(self.encoders, self.attentions)):
            x = enc(x)
            x = att(x)
            enc_features.append(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        
        enc_features = enc_features[::-1]
        x = enc_features[0]
        ds_outputs = []
        
        for i, (up, dec) in enumerate(zip(self.ups, self.dec_convs)):
            x = up(x)
            skip = enc_features[i+1]
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            if self.use_deep_supervision and self.training and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        out = self.final(x)
        if self.use_deep_supervision and self.training:
            return {'output': out, 'deep': ds_outputs}
        return out


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = ResEncUNet3D(features=cfg.FEATURES, n_blocks=cfg.N_BLOCKS)
print(f"Model: {count_params(_m)/1e6:.1f}M params")
print(f"N_BLOCKS: {cfg.N_BLOCKS}")
del _m

In [ ]:
# =============================================================================
# CELL 3: LOSS FUNCTIONS (RELATIVE TO EPOCHS_PER_FOLD)
# =============================================================================

def soft_skeletonize(x, num_iter=10):
    for _ in range(num_iter):
        min_pool = -F.max_pool3d(-x, 3, stride=1, padding=1)
        max_min_pool = F.max_pool3d(min_pool, 3, stride=1, padding=1)
        x = F.relu(x - max_min_pool)
    return x


class CombinedLoss(nn.Module):
    """Dice+BCE with scheduled skeleton/clDice topology terms."""

    def __init__(
        self,
        dice_w=0.45,
        bce_w=0.20,
        skel_w=0.20,
        cldice_w=0.15,
        skel_start=45,
        skel_warmup=45,
        cldice_start=120,
        cldice_warmup=60,
    ):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w = bce_w
        self.skel_w = skel_w
        self.cldice_w = cldice_w

        self.skel_start = skel_start
        self.skel_warmup = skel_warmup
        self.cldice_start = cldice_start
        self.cldice_warmup = cldice_warmup

        self.ds_weights = [0.5, 0.25, 0.125]

    def _scale(self, epoch, start, warmup):
        if epoch < start:
            return 0.0
        if epoch < start + warmup:
            return (epoch - start) / max(warmup, 1)
        return 1.0

    def dice_loss(self, pred, target, mask=None):
        pred = torch.sigmoid(pred)
        if mask is not None:
            valid = (1 - mask)
            pred = pred * valid
            target = target * valid
        inter = (pred * target).sum()
        union = pred.sum() + target.sum()
        return 1 - (2 * inter + 1e-5) / (union + 1e-5)

    def bce_loss(self, pred, target, mask=None):
        if mask is not None:
            valid = 1 - mask
            loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
            return (loss * valid).sum() / (valid.sum() + 1e-8)
        return F.binary_cross_entropy_with_logits(pred, target)

    def skeleton_loss(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            valid = 1 - mask
            pred_sig = pred_sig * valid
            target = target * valid
        target_skel = soft_skeletonize(target, num_iter=10)
        target_tube = F.max_pool3d(target_skel, kernel_size=5, stride=1, padding=2)
        recall = (pred_sig * target_tube).sum() / (target_tube.sum() + 1e-5)
        return 1 - recall

    def cldice_loss(self, pred, target, mask=None):
        pred_sig = torch.sigmoid(pred)
        if mask is not None:
            valid = 1 - mask
            pred_sig = pred_sig * valid
            target = target * valid
        skel_pred = soft_skeletonize(pred_sig, 10)
        skel_target = soft_skeletonize(target, 10)
        tprec = ((skel_pred * target).sum() + 1e-5) / (skel_pred.sum() + 1e-5)
        tsens = ((skel_target * pred_sig).sum() + 1e-5) / (skel_target.sum() + 1e-5)
        return 1 - 2 * tprec * tsens / (tprec + tsens + 1e-5)

    def forward(self, output, target, ignore_mask, epoch):
        if isinstance(output, dict):
            pred = output['output']
            deep = output.get('deep', [])
        else:
            pred = output
            deep = []

        skel_scale = self._scale(epoch, self.skel_start, self.skel_warmup)
        cldice_scale = self._scale(epoch, self.cldice_start, self.cldice_warmup)

        dice = self.dice_loss(pred, target, ignore_mask)
        bce = self.bce_loss(pred, target, ignore_mask)

        skel = self.skeleton_loss(pred, target, ignore_mask) if skel_scale > 0 else torch.tensor(0.0, device=pred.device)
        cldice = self.cldice_loss(pred, target, ignore_mask) if cldice_scale > 0 else torch.tensor(0.0, device=pred.device)

        total = (
            self.dice_w * dice
            + self.bce_w * bce
            + self.skel_w * skel_scale * skel
            + self.cldice_w * cldice_scale * cldice
        )

        for i, ds in enumerate(deep):
            if i >= len(self.ds_weights):
                break
            ds_target = F.interpolate(target, size=ds.shape[2:], mode='nearest')
            ds_mask = F.interpolate(ignore_mask, size=ds.shape[2:], mode='nearest')
            total = total + self.ds_weights[i] * self.dice_loss(ds, ds_target, ds_mask)

        return {
            'total': total,
            'dice': float(dice.detach().item()),
            'bce': float(bce.detach().item()),
            'skel': float(skel.detach().item()) if skel_scale > 0 else 0.0,
            'cldice': float(cldice.detach().item()) if cldice_scale > 0 else 0.0,
            'skel_scale': float(skel_scale),
            'cldice_scale': float(cldice_scale),
        }


print('Loss schedule for EPOCHS_PER_FOLD:', cfg.EPOCHS_PER_FOLD)
print(cfg.SCHEDULE)


In [ ]:
# =============================================================================
# CELL 4: DATASET + STRATIFIED VOLUME FOLDS
# =============================================================================

def load_volume(path: Path) -> np.ndarray:
    try:
        import tifffile
        return tifffile.imread(str(path))
    except Exception:
        im = Image.open(path)
        return np.stack([np.array(p) for p in ImageSequence.Iterator(im)], axis=0)


class VesuviusDatasetCV(Dataset):
    """Volume-level dataset with explicit IDs. Labels remain raw; ignore is lbl==2."""

    def __init__(
        self,
        volume_ids: List[int],
        images_dir: Path,
        labels_dir: Path,
        patch_size=(128, 128, 128),
        is_train=True,
        patches_per_volume=16,
        preload=True,
    ):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.patch_size = patch_size
        self.is_train = is_train
        self.patches_per_volume = patches_per_volume
        self.volume_ids = [int(v) for v in volume_ids]

        self.volumes: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
        self.fg_coords: Dict[int, Optional[np.ndarray]] = {}

        if preload:
            print(f'Preloading {len(self.volume_ids)} volumes...')
            for vid in tqdm(self.volume_ids, desc='Loading'):
                img = load_volume(self.images_dir / f'{vid}.tif').astype(np.float32)
                lbl = load_volume(self.labels_dir / f'{vid}.tif').astype(np.uint8)
                img = (img - img.mean()) / (img.std() + 1e-8)
                self.volumes[vid] = (img, lbl)

                fg = np.argwhere(lbl == 1)
                if len(fg) > 15000:
                    fg = fg[np.random.choice(len(fg), 15000, replace=False)]
                self.fg_coords[vid] = fg if len(fg) > 0 else None

            gb = sum(v[0].nbytes + v[1].nbytes for v in self.volumes.values()) / 1e9
            print(f'Loaded {len(self.volumes)} volumes ({gb:.1f} GB)')

    def __len__(self):
        return len(self.volume_ids) * self.patches_per_volume

    def __getitem__(self, idx):
        vid = self.volume_ids[idx // self.patches_per_volume]
        img, lbl = self.volumes[vid]

        d, h, w = img.shape
        pd, ph, pw = self.patch_size

        if d < pd or h < ph or w < pw:
            img = np.pad(img, ((0, max(0, pd - d)), (0, max(0, ph - h)), (0, max(0, pw - w))), mode='reflect')
            lbl = np.pad(lbl, ((0, max(0, pd - d)), (0, max(0, ph - h)), (0, max(0, pw - w))), mode='constant', constant_values=2)
            d, h, w = img.shape

        fg = self.fg_coords.get(vid)
        if self.is_train and random.random() < cfg.FG_OVERSAMPLE_RATIO and fg is not None and len(fg) > 0:
            c = fg[random.randint(0, len(fg) - 1)]
            z = max(0, min(int(c[0]) - pd // 2, d - pd))
            y = max(0, min(int(c[1]) - ph // 2, h - ph))
            x = max(0, min(int(c[2]) - pw // 2, w - pw))
        else:
            z = random.randint(0, max(0, d - pd))
            y = random.randint(0, max(0, h - ph))
            x = random.randint(0, max(0, w - pw))

        img_p = img[z:z + pd, y:y + ph, x:x + pw].copy()
        lbl_p = lbl[z:z + pd, y:y + ph, x:x + pw].copy()

        if self.is_train:
            for ax in range(3):
                if random.random() > 0.5:
                    img_p = np.flip(img_p, ax)
                    lbl_p = np.flip(lbl_p, ax)
            k = random.randint(0, 3)
            if k:
                img_p = np.rot90(img_p, k, (1, 2))
                lbl_p = np.rot90(lbl_p, k, (1, 2))
            img_p = np.ascontiguousarray(img_p)
            lbl_p = np.ascontiguousarray(lbl_p)
            if random.random() > 0.5:
                img_p = img_p * random.uniform(0.85, 1.15)
            if random.random() > 0.5:
                img_p = img_p + random.uniform(-0.08, 0.08)

        fg_mask = (lbl_p == 1).astype(np.float32)
        ig_mask = (lbl_p == 2).astype(np.float32)

        return {
            'image': torch.from_numpy(img_p).unsqueeze(0).float(),
            'label': torch.from_numpy(fg_mask).unsqueeze(0).float(),
            'ignore_mask': torch.from_numpy(ig_mask).unsqueeze(0).float(),
            'volume_id': int(vid),
        }


def make_stratified_volume_folds(df: pd.DataFrame, n_splits: int = 3, seed: int = 42) -> List[Tuple[List[int], List[int]]]:
    """Public interface requested: stratified folds by scroll_id at volume level."""
    if 'id' not in df.columns or 'scroll_id' not in df.columns:
        raise ValueError('DataFrame must contain id and scroll_id columns')

    ids = df['id'].astype(int).values
    labels = df['scroll_id'].astype(str).values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    folds: List[Tuple[List[int], List[int]]] = []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(ids, labels)):
        train_ids = ids[tr_idx].tolist()
        val_ids = ids[va_idx].tolist()
        assert set(train_ids).isdisjoint(set(val_ids)), f'Fold {fold} has overlap between train and val IDs'
        folds.append((train_ids, val_ids))
    return folds


def prepare_cv_folds(csv_path: Path, images_dir: Path, labels_dir: Path, n_splits: int, seed: int):
    df = pd.read_csv(csv_path)
    valid_mask = [
        (Path(images_dir) / f'{int(i)}.tif').exists() and (Path(labels_dir) / f'{int(i)}.tif').exists()
        for i in df['id'].values
    ]
    valid_df = df.loc[valid_mask].reset_index(drop=True)
    folds = make_stratified_volume_folds(valid_df, n_splits=n_splits, seed=seed)

    manifest = {
        'seed': seed,
        'n_splits': n_splits,
        'n_valid_volumes': int(len(valid_df)),
        'folds': [
            {'fold': f, 'train_ids': tr, 'val_ids': va, 'n_train': len(tr), 'n_val': len(va)}
            for f, (tr, va) in enumerate(folds)
        ],
    }
    cfg.FOLD_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    cfg.FOLD_MANIFEST.write_text(json.dumps(manifest, indent=2))
    print(f'Wrote fold manifest: {cfg.FOLD_MANIFEST}')

    for f, (tr, va) in enumerate(folds):
        print(f'Fold {f}: train={len(tr)} volumes | val={len(va)} volumes')

    return folds, valid_df


print('Dataset + stratified fold utilities ready')


In [ ]:
# =============================================================================
# CELL 5: CHECKPOINT + METRIC IO HELPERS
# =============================================================================

def clean_state_dict(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        k.replace('module.', '').replace('_orig_mod.', ''): v
        for k, v in state_dict.items()
    }


def append_metrics_row(csv_path: Path, row: Dict[str, object]):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not csv_path.exists()
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def save_best_checkpoint(
    fold_dir: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler: GradScaler,
    epoch: int,
    metrics: Dict[str, float],
):
    payload = {
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'metrics': metrics,
        'config': {
            'patch_size': cfg.PATCH_SIZE,
            'features': cfg.FEATURES,
            'n_blocks': cfg.N_BLOCKS,
            'schedule': cfg.SCHEDULE,
            'loss_weights': {
                'dice': cfg.DICE_WEIGHT,
                'bce': cfg.BCE_WEIGHT,
                'skeleton': cfg.SKELETON_WEIGHT,
                'cldice': cfg.CLDICE_WEIGHT,
            },
        },
    }
    fold_dir.mkdir(parents=True, exist_ok=True)
    torch.save(payload, fold_dir / 'best_model.pth')


def load_best_checkpoint_if_exists(model: nn.Module, fold_dir: Path, device: str):
    path = fold_dir / 'best_model.pth'
    if not path.exists():
        return None
    ckpt = torch.load(path, map_location=device, weights_only=False)
    state = clean_state_dict(ckpt['model_state_dict'])
    model.load_state_dict(state, strict=False)
    return ckpt


print('Checkpoint/metrics helpers ready')


In [ ]:
# =============================================================================
# CELL 6: VALIDATION + LEADERBOARD METRICS + FULL-VOLUME INFERENCE
# =============================================================================

try:
    import cc3d
    USE_CC3D = True
except Exception:
    USE_CC3D = False


def connected_components_3d(vol: np.ndarray, connectivity: int = 26) -> np.ndarray:
    if USE_CC3D:
        return cc3d.connected_components(vol.astype(np.uint8), connectivity=connectivity)
    struct = ndimage.generate_binary_structure(3, 3 if connectivity == 26 else 1)
    lb, _ = ndimage.label(vol.astype(bool), structure=struct)
    return lb


def hysteresis_threshold_3d(prob_map: np.ndarray, low: float, high: float) -> np.ndarray:
    high_mask = prob_map >= high
    low_mask = prob_map >= low
    labeled = connected_components_3d(low_mask, connectivity=26)
    keep_labels = np.unique(labeled[high_mask])
    keep_labels = keep_labels[keep_labels > 0]
    return np.isin(labeled, keep_labels).astype(np.uint8)


def apply_postprocess(prob_map: np.ndarray, params: Optional[Dict[str, float]] = None) -> np.ndarray:
    p = {
        'threshold': cfg.PP_THRESHOLD,
        'use_hysteresis': cfg.PP_USE_HYSTERESIS,
        'hysteresis_low': cfg.PP_HYST_LOW,
        'hysteresis_high': cfg.PP_HYST_HIGH,
        'min_component_size': cfg.PP_MIN_COMPONENT_SIZE,
        'closing_iters': cfg.PP_CLOSING_ITERS,
    }
    if params is not None:
        p.update(params)

    if p['use_hysteresis']:
        binary = hysteresis_threshold_3d(prob_map, p['hysteresis_low'], p['hysteresis_high'])
    else:
        binary = (prob_map >= p['threshold']).astype(np.uint8)

    if p['min_component_size'] > 0:
        labeled = connected_components_3d(binary, connectivity=26)
        if labeled.max() > 0:
            result = np.zeros_like(binary, dtype=np.uint8)
            for i in range(1, int(labeled.max()) + 1):
                c = labeled == i
                if c.sum() >= p['min_component_size']:
                    result[c] = 1
            binary = result

    if p['closing_iters'] > 0:
        struct = ndimage.generate_binary_structure(3, 1)
        binary = ndimage.binary_closing(binary, structure=struct, iterations=int(p['closing_iters'])).astype(np.uint8)

    return binary.astype(np.uint8)


@torch.no_grad()
def validate_patch_dice(model: nn.Module, loader: DataLoader, device: str) -> Dict[str, float]:
    model.eval()
    total_dice = 0.0
    n = 0
    for batch in tqdm(loader, desc='ValPatch', file=sys.stdout, leave=False):
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].numpy()
        ignore = batch['ignore_mask'].numpy()
        with autocast(enabled=cfg.USE_AMP):
            out = model(images)
            if isinstance(out, dict):
                out = out['output']
            probs = torch.sigmoid(out).cpu().numpy()
        for b in range(images.shape[0]):
            pred = (probs[b, 0] > 0.5).astype(np.uint8)
            tgt = labels[b, 0].astype(np.uint8)
            ign = ignore[b, 0] > 0.5
            pred[ign] = 0
            tgt[ign] = 0
            inter = (pred & tgt).sum()
            union = pred.sum() + tgt.sum()
            total_dice += (2 * inter + 1e-5) / (union + 1e-5)
            n += 1
    return {'val_patch_dice': float(total_dice / max(n, 1))}


@torch.no_grad()
def infer_volume_prob_map(
    model: nn.Module,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int],
    overlap: float = 0.5,
    tta_mode: str = 'none',
) -> np.ndarray:
    """Sliding-window full-volume inference. tta_mode: 'none' or 'zflip'."""

    def _single_pass(vol: np.ndarray) -> np.ndarray:
        D, H, W = vol.shape
        pd, ph, pw = patch_size
        sd = max(1, int(pd * (1 - overlap)))
        sh = max(1, int(ph * (1 - overlap)))
        sw = max(1, int(pw * (1 - overlap)))

        pred_sum = np.zeros((D, H, W), dtype=np.float32)
        count = np.zeros((D, H, W), dtype=np.float32)

        sigma = 0.125
        gz = np.exp(-0.5 * ((np.arange(pd) - pd / 2) / (pd * sigma)) ** 2)
        gy = np.exp(-0.5 * ((np.arange(ph) - ph / 2) / (ph * sigma)) ** 2)
        gx = np.exp(-0.5 * ((np.arange(pw) - pw / 2) / (pw * sigma)) ** 2)
        gauss = (gz[:, None, None] * gy[None, :, None] * gx[None, None, :]).astype(np.float32)

        z_pos = list(range(0, max(1, D - pd + 1), sd))
        if D > pd and (D - pd) not in z_pos:
            z_pos.append(D - pd)
        y_pos = list(range(0, max(1, H - ph + 1), sh))
        if H > ph and (H - ph) not in y_pos:
            y_pos.append(H - ph)
        x_pos = list(range(0, max(1, W - pw + 1), sw))
        if W > pw and (W - pw) not in x_pos:
            x_pos.append(W - pw)

        vol = (vol.astype(np.float32) - vol.mean()) / (vol.std() + 1e-8)

        model.eval()
        for z in z_pos:
            for y in y_pos:
                for x in x_pos:
                    patch = vol[z:z + pd, y:y + ph, x:x + pw].copy()
                    actual = patch.shape
                    if patch.shape != (pd, ph, pw):
                        patch = np.pad(
                            patch,
                            [(0, pd - patch.shape[0]), (0, ph - patch.shape[1]), (0, pw - patch.shape[2])],
                            mode='reflect',
                        )
                    inp = torch.from_numpy(patch[None, None].astype(np.float32)).to(cfg.DEVICE)
                    with autocast(enabled=cfg.USE_AMP):
                        out = model(inp)
                        if isinstance(out, dict):
                            out = out['output']
                        pred = torch.sigmoid(out).squeeze().cpu().numpy()

                    out_crop = pred[:actual[0], :actual[1], :actual[2]]
                    w_crop = gauss[:actual[0], :actual[1], :actual[2]]
                    pred_sum[z:z + actual[0], y:y + actual[1], x:x + actual[2]] += out_crop * w_crop
                    count[z:z + actual[0], y:y + actual[1], x:x + actual[2]] += w_crop

        return pred_sum / np.maximum(count, 1e-8)

    base = _single_pass(volume)
    if tta_mode == 'zflip':
        flip = _single_pass(volume[::-1].copy())[::-1].copy()
        base = 0.5 * (base + flip)
    return base.astype(np.float32)


def _surface_dice_surrogate(pred_bin: np.ndarray, gt_bin: np.ndarray, tolerance: float = 2.0) -> float:
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)
    if not pred.any() and not gt.any():
        return 1.0
    if pred.any() ^ gt.any():
        return 0.0

    struct = ndimage.generate_binary_structure(3, 1)
    pred_surf = pred ^ ndimage.binary_erosion(pred, structure=struct, border_value=0)
    gt_surf = gt ^ ndimage.binary_erosion(gt, structure=struct, border_value=0)

    dist_to_gt = distance_transform_edt(~gt_surf)
    dist_to_pred = distance_transform_edt(~pred_surf)

    pred_hit = pred_surf & (dist_to_gt <= tolerance)
    gt_hit = gt_surf & (dist_to_pred <= tolerance)
    denom = pred_surf.sum() + gt_surf.sum() + 1e-8
    return float((pred_hit.sum() + gt_hit.sum()) / denom)


def _voi_score_surrogate(pred_bin: np.ndarray, gt_bin: np.ndarray, connectivity: int = 26, alpha: float = 0.3):
    pred = pred_bin.astype(bool)
    gt = gt_bin.astype(bool)
    if not pred.any() and not gt.any():
        return 1.0, 0.0, 0.0

    union = pred | gt
    if union.sum() == 0:
        return 1.0, 0.0, 0.0

    pred_lab = connected_components_3d(pred, connectivity=connectivity).astype(np.int64)
    gt_lab = connected_components_3d(gt, connectivity=connectivity).astype(np.int64)

    p = pred_lab[union]
    g = gt_lab[union]
    max_p = int(p.max()) + 1
    max_g = int(g.max()) + 1
    flat = p * max_g + g
    counts = np.bincount(flat, minlength=max_p * max_g).reshape(max_p, max_g).astype(np.float64)

    n = counts.sum() + 1e-12
    px = counts.sum(axis=1) / n
    py = counts.sum(axis=0) / n
    pxy = counts / n

    nzx = px > 0
    nzy = py > 0
    nzz = pxy > 0

    Hx = -np.sum(px[nzx] * np.log(px[nzx]))
    Hy = -np.sum(py[nzy] * np.log(py[nzy]))
    Hxy = -np.sum(pxy[nzz] * np.log(pxy[nzz]))

    voi_split = max(0.0, Hxy - Hx)   # H(GT | Pred)
    voi_merge = max(0.0, Hxy - Hy)   # H(Pred | GT)
    voi_total = voi_split + voi_merge
    voi_score = 1.0 / (1.0 + alpha * voi_total)
    return float(voi_score), float(voi_split), float(voi_merge)


def _load_topometrics_compute_fn():
    root = Path('/kaggle/working/topological-metrics-kaggle/src')
    local_root = Path('/Users/manishswami/developer/Vesuvius-inference/notebooks/archive/topological-metrics-kaggle/src')
    src = root if root.exists() else local_root
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
    try:
        from topometrics import compute_leaderboard_score
        return compute_leaderboard_score, None
    except Exception as e:
        return None, str(e)


_TOPOMETRICS_FN, _TOPOMETRICS_ERR = _load_topometrics_compute_fn()
if _TOPOMETRICS_FN is None:
    print('Topometrics full metric unavailable, using SurfaceDice+VOI surrogate for ranking.')
    print('Reason:', _TOPOMETRICS_ERR)
else:
    print('Topometrics full metric available.')


def evaluate_leaderboard(pred: np.ndarray, gt: np.ndarray, ignore_label: int = 2) -> Dict[str, float]:
    """Public interface requested. Accepts binary or prob prediction + raw GT labels."""

    pred_arr = pred.astype(np.float32)
    if pred_arr.max() > 1.0 or pred_arr.min() < 0.0:
        pred_bin = (pred_arr > 0).astype(np.uint8)
    else:
        pred_bin = (pred_arr > 0.5).astype(np.uint8)

    gt_fg = (gt == 1).astype(np.uint8)
    ignore = (gt == ignore_label)
    pred_bin[ignore] = 0
    gt_fg[ignore] = 0

    if _TOPOMETRICS_FN is not None:
        rep = _TOPOMETRICS_FN(
            predictions=pred_bin,
            labels=gt,
            dims=(0, 1, 2),
            spacing=cfg.SPACING,
            surface_tolerance=cfg.SURFACE_TOLERANCE,
            voi_connectivity=cfg.VOI_CONNECTIVITY,
            combine_weights=(0.30, 0.35, 0.35),
            ignore_label=ignore_label,
            fg_threshold=None,
        )
        return {
            'selection_score': float(rep.score),
            'lb_score': float(rep.score),
            'topo_score': float(rep.topo.toposcore),
            'surface_dice': float(rep.surface_dice),
            'voi_score': float(rep.voi.voi_score),
            'voi_split': float(rep.voi.voi_split),
            'voi_merge': float(rep.voi.voi_merge),
            'fallback_used': 0.0,
        }

    sd = _surface_dice_surrogate(pred_bin, gt_fg, tolerance=cfg.SURFACE_TOLERANCE)
    voi_score, voi_split, voi_merge = _voi_score_surrogate(pred_bin, gt_fg, connectivity=cfg.VOI_CONNECTIVITY, alpha=0.3)
    surrogate = 0.5 * sd + 0.5 * voi_score
    return {
        'selection_score': float(surrogate),
        'lb_score': float('nan'),
        'topo_score': float('nan'),
        'surface_dice': float(sd),
        'voi_score': float(voi_score),
        'voi_split': float(voi_split),
        'voi_merge': float(voi_merge),
        'fallback_used': 1.0,
    }


@torch.no_grad()
def evaluate_fold_leaderboard(
    model: nn.Module,
    dataset: VesuviusDatasetCV,
    val_ids: List[int],
    max_cases: int,
    postprocess_params: Optional[Dict[str, float]] = None,
):
    chosen_ids = [int(v) for v in val_ids[:max_cases]]
    rows = []
    for vid in chosen_ids:
        vol, gt = dataset.volumes[vid]
        prob = infer_volume_prob_map(
            model,
            vol,
            patch_size=cfg.PATCH_SIZE,
            overlap=cfg.VALIDATION_OVERLAP,
            tta_mode='none',
        )
        pred = apply_postprocess(prob, params=postprocess_params)
        m = evaluate_leaderboard(pred, gt, ignore_label=cfg.IGNORE_LABEL)
        m['volume_id'] = int(vid)
        rows.append(m)

    if not rows:
        return {
            'selection_score': float('-inf'),
            'lb_score': float('nan'),
            'surface_dice': float('nan'),
            'voi_score': float('nan'),
            'topo_score': float('nan'),
            'fallback_used': 1.0,
            'n_cases': 0,
        }, pd.DataFrame([])

    df = pd.DataFrame(rows)
    summary = {
        'selection_score': float(df['selection_score'].mean()),
        'lb_score': float(df['lb_score'].mean()) if 'lb_score' in df else float('nan'),
        'surface_dice': float(df['surface_dice'].mean()),
        'voi_score': float(df['voi_score'].mean()),
        'topo_score': float(df['topo_score'].mean()) if 'topo_score' in df else float('nan'),
        'fallback_used': float(df['fallback_used'].max()),
        'n_cases': int(len(df)),
    }
    return summary, df


print('Validation + metric functions ready')


In [ ]:
# =============================================================================
# CELL 10: 3-FOLD CV TRAINING + OOF ARTIFACTS
# =============================================================================

def _assert_split_integrity(folds: List[Tuple[List[int], List[int]]]):
    all_val = []
    for i, (tr, va) in enumerate(folds):
        tr_set, va_set = set(tr), set(va)
        assert tr_set.isdisjoint(va_set), f'Fold {i} has train/val overlap'
        all_val.extend(va)
    dup_vals = pd.Series(all_val).value_counts()
    overlap_vals = dup_vals[dup_vals > 1]
    if not overlap_vals.empty:
        raise AssertionError(f'Validation ID appears in multiple folds: {overlap_vals.head().to_dict()}')


def _log_schedule_checkpoints(epoch: int, losses: Dict[str, float], schedule: Dict[str, int]):
    checkpoints = {0, 45, 120, 180, 260, schedule['skel_start'], schedule['cldice_start']}
    if epoch in checkpoints:
        print(
            f'[ScheduleCheck] epoch={epoch:03d} '
            f"skel_scale={losses.get('skel_scale', 0):.3f} "
            f"cldice_scale={losses.get('cldice_scale', 0):.3f}"
        )


@torch.no_grad()
def save_oof_predictions(model: nn.Module, dataset: VesuviusDatasetCV, val_ids: List[int], fold_oof_dir: Path):
    fold_oof_dir.mkdir(parents=True, exist_ok=True)
    records = []
    for vid in tqdm(val_ids, desc='OOF', leave=False):
        vid = int(vid)
        vol, gt = dataset.volumes[vid]
        prob = infer_volume_prob_map(model, vol, patch_size=cfg.PATCH_SIZE, overlap=0.5, tta_mode='none')
        np.save(fold_oof_dir / f'{vid}_prob.npy', prob.astype(np.float16))
        pred = apply_postprocess(prob)
        m = evaluate_leaderboard(pred, gt, ignore_label=cfg.IGNORE_LABEL)
        m['volume_id'] = vid
        records.append(m)

    oof_df = pd.DataFrame(records)
    oof_df.to_csv(fold_oof_dir / 'oof_metrics.csv', index=False)
    return oof_df


def train_fold(fold: int, train_ids: List[int], val_ids: List[int], epochs: int = 300):
    print('=' * 90)
    print(f'FOLD {fold} TRAINING')
    print('=' * 90)

    schedule = build_epoch_schedule(epochs)
    train_images = cfg.DATA_ROOT / 'train_images'
    train_labels = cfg.DATA_ROOT / 'train_labels'

    train_ds = VesuviusDatasetCV(
        train_ids,
        train_images,
        train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=True,
        patches_per_volume=cfg.PATCHES_PER_VOLUME,
    )
    val_patch_ds = VesuviusDatasetCV(
        val_ids,
        train_images,
        train_labels,
        patch_size=cfg.PATCH_SIZE,
        is_train=False,
        patches_per_volume=cfg.VAL_PATCHES_PER_VOLUME,
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=cfg.PREFETCH_FACTOR,
    )
    val_patch_loader = DataLoader(
        val_patch_ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
    )

    model = ResEncUNet3D(
        features=cfg.FEATURES,
        n_blocks=cfg.N_BLOCKS,
        use_scse=cfg.USE_SCSE,
        use_deep_supervision=cfg.USE_DEEP_SUPERVISION,
    ).to(cfg.DEVICE)

    if hasattr(torch, 'compile'):
        model = torch.compile(model, mode='reduce-overhead')

    criterion = CombinedLoss(
        dice_w=cfg.DICE_WEIGHT,
        bce_w=cfg.BCE_WEIGHT,
        skel_w=cfg.SKELETON_WEIGHT,
        cldice_w=cfg.CLDICE_WEIGHT,
        skel_start=schedule['skel_start'],
        skel_warmup=schedule['skel_warmup'],
        cldice_start=schedule['cldice_start'],
        cldice_warmup=schedule['cldice_warmup'],
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LEARNING_RATE,
        betas=(cfg.BETA1, cfg.BETA2),
        weight_decay=cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
        eta_min=cfg.ETA_MIN,
    )
    scaler = GradScaler(enabled=cfg.USE_AMP)

    fold_dir = cfg.CHECKPOINT_DIR / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    metrics_csv = fold_dir / 'metrics.csv'

    best_selection = -np.inf
    best_epoch = -1
    best_metrics = {}
    no_improve_epochs = 0

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        total_loss = 0.0
        total_dice_loss = 0.0
        n_batches = 0

        pbar = tqdm(train_loader, desc=f'F{fold} E{epoch+1}/{epochs}', file=sys.stdout, leave=False)
        last_losses = None
        for batch in pbar:
            images = batch['image'].to(cfg.DEVICE, non_blocking=True)
            labels = batch['label'].to(cfg.DEVICE, non_blocking=True)
            ignore = batch['ignore_mask'].to(cfg.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=cfg.USE_AMP):
                out = model(images)
                losses = criterion(out, labels, ignore, epoch)

            scaler.scale(losses['total']).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()

            total_loss += losses['total'].item()
            total_dice_loss += losses['dice']
            n_batches += 1
            last_losses = losses

            pbar.set_postfix(
                loss=f"{losses['total'].item():.3f}",
                d=f"{losses['dice']:.3f}",
                sk=f"{losses['skel_scale']:.2f}",
                cl=f"{losses['cldice_scale']:.2f}",
            )

        scheduler.step()

        if last_losses is not None:
            _log_schedule_checkpoints(epoch, last_losses, schedule)

        train_metrics = {
            'train_loss': float(total_loss / max(n_batches, 1)),
            'train_dice_loss': float(total_dice_loss / max(n_batches, 1)),
        }

        patch_dice = float('nan')
        if (epoch + 1) % cfg.PATCH_DICE_EVERY == 0:
            patch_dice = validate_patch_dice(model, val_patch_loader, cfg.DEVICE)['val_patch_dice']

        lb_summary = {
            'selection_score': float('nan'),
            'lb_score': float('nan'),
            'surface_dice': float('nan'),
            'voi_score': float('nan'),
            'topo_score': float('nan'),
            'fallback_used': float('nan'),
            'n_cases': 0,
        }

        if (epoch + 1) % cfg.LB_EVAL_EVERY == 0:
            lb_summary, lb_cases = evaluate_fold_leaderboard(
                model,
                val_patch_ds,
                val_ids=val_ids,
                max_cases=min(cfg.LB_EVAL_MAX_CASES, len(val_ids)),
            )
            lb_cases.to_csv(fold_dir / f'lb_eval_epoch_{epoch+1:04d}.csv', index=False)

            sel = lb_summary['selection_score']
            if np.isfinite(sel) and sel > best_selection:
                best_selection = sel
                best_epoch = epoch
                no_improve_epochs = 0
                best_metrics = {
                    **train_metrics,
                    'val_patch_dice': patch_dice,
                    **lb_summary,
                }
                save_best_checkpoint(
                    fold_dir,
                    model,
                    optimizer,
                    scheduler,
                    scaler,
                    epoch,
                    best_metrics,
                )
            else:
                no_improve_epochs += cfg.LB_EVAL_EVERY
        else:
            no_improve_epochs += 1

        row = {
            'fold': int(fold),
            'epoch': int(epoch + 1),
            'lr': float(scheduler.get_last_lr()[0]),
            **train_metrics,
            'val_patch_dice': patch_dice,
            **lb_summary,
            'best_selection_so_far': float(best_selection) if np.isfinite(best_selection) else float('nan'),
            'best_epoch_so_far': int(best_epoch + 1) if best_epoch >= 0 else -1,
            'elapsed_sec': float(time.time() - t0),
        }
        append_metrics_row(metrics_csv, row)

        msg = (
            f"Fold {fold} | Epoch {epoch+1}/{epochs} | "
            f"loss={row['train_loss']:.4f} | patch_dice={patch_dice:.4f}"
            if np.isfinite(patch_dice)
            else f"Fold {fold} | Epoch {epoch+1}/{epochs} | loss={row['train_loss']:.4f}"
        )
        if np.isfinite(lb_summary['selection_score']):
            msg += f" | selection={lb_summary['selection_score']:.4f}"
            if np.isfinite(lb_summary['lb_score']):
                msg += f" | lb={lb_summary['lb_score']:.4f}"
        print(msg)

        if no_improve_epochs >= cfg.EARLY_STOP_PATIENCE:
            print(f'Early stopping fold {fold} at epoch {epoch+1} (patience {cfg.EARLY_STOP_PATIENCE})')
            break

    if best_epoch < 0:
        raise RuntimeError(f'Fold {fold} never produced a valid selection score')

    # Load best model and generate OOF probability maps
    _ = load_best_checkpoint_if_exists(model, fold_dir, cfg.DEVICE)
    fold_oof_dir = cfg.OOF_DIR / f'fold_{fold}'
    oof_df = save_oof_predictions(model, val_patch_ds, val_ids, fold_oof_dir)

    summary = {
        'fold': int(fold),
        'best_epoch': int(best_epoch + 1),
        'best_selection_score': float(best_metrics.get('selection_score', float('nan'))),
        'best_lb_score': float(best_metrics.get('lb_score', float('nan'))),
        'best_patch_dice': float(best_metrics.get('val_patch_dice', float('nan'))),
        'best_surface_dice': float(best_metrics.get('surface_dice', float('nan'))),
        'best_voi_score': float(best_metrics.get('voi_score', float('nan'))),
        'best_topo_score': float(best_metrics.get('topo_score', float('nan'))),
        'fallback_used': float(best_metrics.get('fallback_used', float('nan'))),
        'n_train_ids': int(len(train_ids)),
        'n_val_ids': int(len(val_ids)),
        'oof_selection_mean': float(oof_df['selection_score'].mean()) if not oof_df.empty else float('nan'),
        'oof_lb_mean': float(oof_df['lb_score'].mean()) if 'lb_score' in oof_df else float('nan'),
    }

    with open(fold_dir / 'fold_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    print(f"Fold {fold} complete. best_selection={summary['best_selection_score']:.4f}, best_lb={summary['best_lb_score']}")
    return summary


def run_kfold_cv(n_splits: int = 3, epochs: int = 300):
    train_csv = cfg.DATA_ROOT / 'train.csv'
    train_images = cfg.DATA_ROOT / 'train_images'
    train_labels = cfg.DATA_ROOT / 'train_labels'

    folds, valid_df = prepare_cv_folds(
        csv_path=train_csv,
        images_dir=train_images,
        labels_dir=train_labels,
        n_splits=n_splits,
        seed=cfg.SEED,
    )

    _assert_split_integrity(folds)

    fold_summaries = []
    for fold, (train_ids, val_ids) in enumerate(folds):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        fold_summary = train_fold(fold=fold, train_ids=train_ids, val_ids=val_ids, epochs=epochs)
        fold_summaries.append(fold_summary)

    summary_df = pd.DataFrame(fold_summaries).sort_values('best_selection_score', ascending=False)
    cfg.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    summary_csv = cfg.METRICS_DIR / 'fold_scores.csv'
    summary_df.to_csv(summary_csv, index=False)

    print('=' * 90)
    print('K-FOLD CV COMPLETE')
    print('=' * 90)
    print(summary_df)
    print(f'Saved: {summary_csv}')

    return summary_df


print('3-fold CV trainer ready.')
print('Usage: run_kfold_cv(n_splits=3, epochs=300)')


In [ ]:
# =============================================================================
# CELL 7: STRICT ENTRYPOINT (NON-CV TRAINING DISABLED)
# =============================================================================

def run_strict_cv_training(n_splits: int = None, epochs_per_fold: int = None):
    if cfg.ENABLE_NON_CV_ENTRYPOINT:
        raise RuntimeError('This notebook is configured for strict CV only. Set ENABLE_NON_CV_ENTRYPOINT=False.')
    n_splits = n_splits or cfg.N_SPLITS
    epochs_per_fold = epochs_per_fold or cfg.EPOCHS_PER_FOLD
    return run_kfold_cv(n_splits=n_splits, epochs=epochs_per_fold)


print('Strict CV entrypoint ready: run_strict_cv_training()')
print('Non-CV main() training has been intentionally removed.')


In [ ]:
# =============================================================================
# CELL 8: RUN TRAINING (MANUAL EXECUTION)
# =============================================================================

# Uncomment to start full CV training.
# fold_results = run_strict_cv_training(n_splits=cfg.N_SPLITS, epochs_per_fold=cfg.EPOCHS_PER_FOLD)
# fold_results

print('Manual execution cell ready.')
print('Use: fold_results = run_strict_cv_training(...)')


In [ ]:
# =============================================================================
# CELL 9: CHECK CV STATUS
# =============================================================================

def check_cv_status():
    print('=' * 80)
    print('CV STATUS')
    print('=' * 80)

    if cfg.FOLD_MANIFEST.exists():
        man = json.loads(cfg.FOLD_MANIFEST.read_text())
        print('Fold manifest:', cfg.FOLD_MANIFEST)
        print('n_valid_volumes:', man.get('n_valid_volumes'))
        print('n_splits:', man.get('n_splits'))
    else:
        print('Fold manifest not found:', cfg.FOLD_MANIFEST)

    fold_dirs = sorted(cfg.CHECKPOINT_DIR.glob('fold_*'))
    if not fold_dirs:
        print('No fold checkpoints yet in', cfg.CHECKPOINT_DIR)
        return

    for fd in fold_dirs:
        best = fd / 'best_model.pth'
        metrics_csv = fd / 'metrics.csv'
        oof_dir = cfg.OOF_DIR / fd.name
        print('-' * 80)
        print(fd.name)
        print(' best_model:', best.exists())
        print(' metrics.csv:', metrics_csv.exists())
        print(' oof_dir:', oof_dir.exists())
        if best.exists():
            ckpt = torch.load(best, map_location='cpu', weights_only=False)
            m = ckpt.get('metrics', {})
            print(' best_epoch:', int(ckpt.get('epoch', -1)) + 1)
            print(' best_selection_score:', m.get('selection_score'))
            print(' best_lb_score:', m.get('lb_score'))
            print(' best_patch_dice:', m.get('val_patch_dice'))

    summary_csv = cfg.METRICS_DIR / 'fold_scores.csv'
    if summary_csv.exists():
        print('-' * 80)
        print('Summary:', summary_csv)
        print(pd.read_csv(summary_csv))


check_cv_status()


In [ ]:
# =============================================================================
# CELL 11: LOAD TOP-3 FOLDS + WEIGHTED ENSEMBLE INFERENCE
# =============================================================================

def load_top_fold_models(checkpoint_dir: Path, top_k: int = 3, device: str = 'cuda'):
    score_csv = cfg.METRICS_DIR / 'fold_scores.csv'
    if score_csv.exists():
        score_df = pd.read_csv(score_csv)
    else:
        rows = []
        for fd in sorted(Path(checkpoint_dir).glob('fold_*')):
            idx = int(fd.name.split('_')[-1])
            summ = fd / 'fold_summary.json'
            if summ.exists():
                rows.append(json.loads(summ.read_text()))
            else:
                rows.append({'fold': idx, 'best_selection_score': 0.0, 'best_lb_score': float('nan')})
        score_df = pd.DataFrame(rows)

    rank_col = 'best_lb_score' if 'best_lb_score' in score_df.columns and score_df['best_lb_score'].notna().any() else 'best_selection_score'
    score_df = score_df.sort_values(rank_col, ascending=False).reset_index(drop=True)
    score_df = score_df.head(top_k).copy()

    raw_scores = score_df[rank_col].fillna(0).values.astype(np.float64)
    if np.all(raw_scores <= 0):
        weights = np.ones(len(score_df), dtype=np.float64) / max(len(score_df), 1)
    else:
        raw_scores = np.clip(raw_scores, a_min=0, a_max=None)
        weights = raw_scores / raw_scores.sum()

    models = []
    used_folds = []
    used_weights = []

    for (_, row), w in zip(score_df.iterrows(), weights):
        fold = int(row['fold'])
        ckpt_path = Path(checkpoint_dir) / f'fold_{fold}' / 'best_model.pth'
        if not ckpt_path.exists():
            print(f'Warning: missing checkpoint for fold {fold}: {ckpt_path}')
            continue

        model = ResEncUNet3D(
            features=cfg.FEATURES,
            n_blocks=cfg.N_BLOCKS,
            use_scse=cfg.USE_SCSE,
            use_deep_supervision=False,
        ).to(device)

        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        state = clean_state_dict(ckpt['model_state_dict'])
        model.load_state_dict(state, strict=False)
        model.eval()

        models.append(model)
        used_folds.append(fold)
        used_weights.append(float(w))

    if not models:
        raise RuntimeError('No fold models could be loaded')

    w = np.array(used_weights, dtype=np.float64)
    w = w / w.sum()
    print('Loaded folds:', used_folds)
    print('Weights:', [float(x) for x in w])
    return models, [float(x) for x in w], score_df


@torch.no_grad()
def weighted_ensemble_inference(
    models: List[nn.Module],
    weights: List[float],
    volume: np.ndarray,
    patch_size=(128, 128, 128),
    overlap=0.5,
    tta_mode='none',
):
    weights = np.array(weights, dtype=np.float64)
    weights = weights / weights.sum()

    ensemble = np.zeros(volume.shape, dtype=np.float32)
    for model, w in zip(models, weights):
        prob = infer_volume_prob_map(model, volume, patch_size=patch_size, overlap=overlap, tta_mode=tta_mode)
        ensemble += (w * prob).astype(np.float32)
    return ensemble


print('Top-3 weighted ensemble utilities ready')


In [ ]:
# =============================================================================
# CELL 12: POSTPROCESS SEARCH + METRIC SMOKE TEST
# =============================================================================

def load_oof_arrays(oof_dir: Path, labels_dir: Path):
    probs, gts = {}, {}
    oof_dir = Path(oof_dir)
    labels_dir = Path(labels_dir)

    for p in sorted(oof_dir.glob('*_prob.npy')):
        vid = int(p.stem.replace('_prob', ''))
        probs[vid] = np.load(p).astype(np.float32)
        gt_path = labels_dir / f'{vid}.tif'
        if gt_path.exists():
            gts[vid] = load_volume(gt_path).astype(np.uint8)

    common = sorted(set(probs).intersection(gts))
    probs = {k: probs[k] for k in common}
    gts = {k: gts[k] for k in common}
    return probs, gts


def search_postprocess_params(oof_probs, oof_gts) -> Dict[str, float]:
    """Public interface requested: grid-search postprocess on composite metric."""

    if isinstance(oof_probs, dict):
        ids = sorted(set(oof_probs).intersection(set(oof_gts)))
        probs = [oof_probs[i] for i in ids]
        gts = [oof_gts[i] for i in ids]
    else:
        probs = list(oof_probs)
        gts = list(oof_gts)
        ids = list(range(len(probs)))

    if len(probs) == 0:
        raise ValueError('No OOF predictions provided')

    grid = {
        'threshold': [0.28, 0.30, 0.32],
        'hysteresis_low': [0.14, 0.16, 0.18],
        'hysteresis_high': [0.36, 0.38, 0.40],
        'min_component_size': [30, 50, 80],
        'closing_iters': [0, 1],
    }

    rows = []
    best_score = -np.inf
    best_params = None

    for thr in grid['threshold']:
        for low in grid['hysteresis_low']:
            for high in grid['hysteresis_high']:
                if low >= high:
                    continue
                for min_size in grid['min_component_size']:
                    for closing in grid['closing_iters']:
                        params = {
                            'threshold': thr,
                            'use_hysteresis': True,
                            'hysteresis_low': low,
                            'hysteresis_high': high,
                            'min_component_size': int(min_size),
                            'closing_iters': int(closing),
                        }

                        case_scores = []
                        case_lb = []
                        case_surf = []
                        case_voi = []
                        case_topo = []
                        fallback_flags = []

                        for prob, gt in zip(probs, gts):
                            pred = apply_postprocess(prob, params=params)
                            m = evaluate_leaderboard(pred, gt, ignore_label=cfg.IGNORE_LABEL)
                            case_scores.append(m['selection_score'])
                            case_lb.append(m['lb_score'])
                            case_surf.append(m['surface_dice'])
                            case_voi.append(m['voi_score'])
                            case_topo.append(m['topo_score'])
                            fallback_flags.append(m['fallback_used'])

                        score = float(np.nanmean(case_scores))
                        row = {
                            **params,
                            'selection_score_mean': score,
                            'lb_mean': float(np.nanmean(case_lb)),
                            'surface_dice_mean': float(np.nanmean(case_surf)),
                            'voi_mean': float(np.nanmean(case_voi)),
                            'topo_mean': float(np.nanmean(case_topo)),
                            'fallback_used': float(np.nanmax(fallback_flags)),
                        }
                        rows.append(row)

                        if score > best_score:
                            best_score = score
                            best_params = params.copy()

    results_df = pd.DataFrame(rows).sort_values('selection_score_mean', ascending=False)
    cfg.METRICS_DIR.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(cfg.METRICS_DIR / 'postprocess_grid_search.csv', index=False)

    payload = {
        'best_params': best_params,
        'best_selection_score': float(best_score),
        'grid_path': str(cfg.METRICS_DIR / 'postprocess_grid_search.csv'),
    }
    with open(cfg.METRICS_DIR / 'best_postprocess.json', 'w') as f:
        json.dump(payload, f, indent=2)

    print('Best postprocess params:', payload)
    return best_params


def metric_smoke_test():
    """Metric smoke test: one synthetic pred/gt pair through evaluation pipeline."""
    gt = np.zeros((32, 32, 32), dtype=np.uint8)
    gt[8:24, 8:24, 8:24] = 1
    gt[:2] = 2

    pred = np.zeros_like(gt, dtype=np.uint8)
    pred[9:25, 9:25, 9:25] = 1
    pred[:2] = 0

    m = evaluate_leaderboard(pred, gt, ignore_label=2)
    print('Metric smoke test:', m)


print('Postprocess search + smoke test ready')
print('Interfaces now available:')
print('  - make_stratified_volume_folds(df, n_splits=3, seed=42)')
print('  - build_epoch_schedule(total_epochs)')
print('  - evaluate_leaderboard(pred, gt, ignore_label=2)')
print('  - search_postprocess_params(oof_probs, oof_gts)')
